# Module 2: Evaluate the scoring

In Module 1 the agent scored transactions and decided when to ask a human. But are its scores any good? In this
module you check them against reference answers from experienced analysts, using LangSmith evaluations.

We measure three things:

| Evaluator | Question | Score |
|-----------|----------|-------|
| `decision_accuracy` | Did the agent approve or block the same transactions as the analysts? | Share that match |
| `score_alignment` | How close are the agent's risk scores to the analysts' scores? | 1 = identical, 0 = 100 points apart |
| `review_routing` | Did the agent ask a human on exactly the transactions analysts found ambiguous? This includes the confidence checks from Module 1. | Share routed correctly |



## Step 1: Build a dataset (3 min)

A LangSmith dataset is a list of examples. Each one has **inputs** and **reference outputs**.

Our agent's decisions depend on memory, so we can't score transactions one at a time. Instead, one example is a
customer's **whole stream**:

- **Inputs:** the customer, their transactions, and what a reviewer says if the agent asks for help.
- **Reference outputs:** the analysts' label for every transaction.

All three come from the JSONL tables in `data/`: `transactions`, `reviewer_answers` and `analyst_labels`.

The dataset name ends with your `ATTENDEE_NAME` from `.env` (or your computer's username), so it doesn't clash
with other attendees' datasets in a shared workspace.

In [ ]:
# Notebook only: let Python find `agent.py` and the `utils/`, `tools/` and `evals/` folders.
# This notebook lives in modules/, so the workshop root is one level up.
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from langsmith import Client

from utils import db
from utils.attendee import ATTENDEE
from utils.state import OCT_19
from utils.transaction_processing import transaction_stream

DATASET_NAME = f"transaction-monitoring-{ATTENDEE}"
REVIEWER_ANSWERS = {row["transaction_id"]: row for row in db.table("reviewer_answers")}

def _build_dataset_examples() -> list[dict]:
    """One example per customer: their stream and reviewer answers as inputs, the analysts' labels as outputs."""
    examples = []
    for customer in db.table("customer_profiles"):
        transactions = transaction_stream(customer["user_id"], end=OCT_19)
        ids = [tx["transaction_id"] for tx in transactions]
        examples.append(
            {
                "inputs": {
                    "user_id": customer["user_id"],
                    "transactions": transactions,
                    "reviewer_answers": {tx_id: REVIEWER_ANSWERS[tx_id] for tx_id in ids if tx_id in REVIEWER_ANSWERS},
                },
                "outputs": {"labels": {tx_id: db.get("analyst_labels", transaction_id=tx_id) for tx_id in ids}},
            }
        )
    return examples

def _create_dataset(client: Client) -> None:
    """Create the LangSmith dataset if this attendee doesn't have one yet."""
    if not client.has_dataset(dataset_name=DATASET_NAME):
        dataset = client.create_dataset(DATASET_NAME, description="Customer transaction streams with analyst labels")
        client.create_examples(dataset_id=dataset.id, examples=_build_dataset_examples())

_create_dataset(Client())
print(f"Dataset '{DATASET_NAME}' is ready. Open Datasets & Experiments in LangSmith to see it.")

## Step 2: The target function (2 min)

`evaluate` calls a **target function** once for each example. Ours builds a fresh agent with empty memory, runs it
over the customer's stream, and returns the decisions from the audit log.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from agent import build_agent
from utils.state import MyAgentContext
from utils.transaction_processing import process_transactions, decision_log

def run_agent_on_stream(context_settings: MyAgentContext, inputs: dict) -> dict:
    """The target function: run a fresh agent over one customer's stream and return its decisions."""
    agent = build_agent(checkpointer=InMemorySaver(), store=InMemoryStore() )

    process_transactions(
        agent, context_settings=context_settings, transactions=inputs["transactions"], thread_id=inputs["user_id"], reviewer_answers=inputs["reviewer_answers"]
    )

    return {"decisions": decision_log(agent, inputs["user_id"])}

## Step 3: Evaluators (4 min)

An evaluator is a plain Python function. LangSmith passes it the target's `outputs` and the example's
`reference_outputs`, and it returns a score. A transaction the agent never recorded counts as wrong.

In [ ]:
def _decisions_by_id(outputs: dict) -> dict:
    return {decision["transaction_id"]: decision for decision in outputs["decisions"]}

def decision_accuracy_eval(outputs: dict, reference_outputs: dict) -> float:
    """Share of transactions where the agent's own decision matches the analysts'."""
    decisions = _decisions_by_id(outputs)
    labels = reference_outputs["labels"]
    matches = [
        tx_id in decisions and decisions[tx_id]["agent_decision"] == label["decision"]
        for tx_id, label in labels.items()
    ]
    return sum(matches) / len(matches)

def score_alignment_eval(outputs: dict, reference_outputs: dict) -> float:
    """1 minus the average gap between the agent's and the analysts' risk scores, on a 0-1 scale."""
    decisions = _decisions_by_id(outputs)
    labels = reference_outputs["labels"]
    gaps = [
        abs(decisions[tx_id]["risk_score"] - label["risk_score"]) if tx_id in decisions else 100
        for tx_id, label in labels.items()
    ]
    return 1 - (sum(gaps) / len(gaps) / 100)

def review_routing_eval(outputs: dict, reference_outputs: dict) -> float:
    """Share of transactions where the agent asked a human exactly when analysts said it should."""
    decisions = _decisions_by_id(outputs)
    labels = reference_outputs["labels"]
    correct = [
        tx_id in decisions and (decisions[tx_id]["decided_by"] == "human") == label["ambiguous"]
        for tx_id, label in labels.items()
    ]
    return sum(correct) / len(correct)


Because evaluators are plain functions, you can try them on a made-up result. Here the agent approved the
Singapore jewellery with a score of 60, when analysts would block it at 95:

In [ ]:
made_up = {
    "decisions": [
        {
            "transaction_id": "TX-2004",
            "risk_score": 60,
            "confidence": 0.9,
            "agent_decision": "approve",
            "decided_by": "agent",
        }
    ]
}
reference = {"labels": {"TX-2004": db.get("analyst_labels", transaction_id="TX-2004")}}
print("decision_accuracy:", decision_accuracy_eval(made_up, reference))
print("score_alignment:  ", score_alignment_eval(made_up, reference))
print("review_routing:   ", review_routing_eval(made_up, reference))

## Step 4: Run the evaluation (4 min)

`evaluate` runs the target on every example, scores the results, and saves everything to LangSmith as an
**experiment**. Open the link it prints to see each stream, the agent's trace, and its scores.

In [ ]:
from langsmith import evaluate

EVALUATORS = [decision_accuracy_eval, score_alignment_eval, review_routing_eval]

def average_scores(results) -> dict:
    """Average each evaluator's score across all the examples in an experiment."""
    scores = {}
    for row in results:
        for result in row["evaluation_results"]["results"]:
            scores.setdefault(result.key, []).append(result.score)

    return {key: round(sum(values) / len(values), 2) for key, values in scores.items()}
    
def run_experiment(prefix: str, context_settings: MyAgentContext) -> dict:
    """Run the evaluation once and return the average scores. The full experiment is saved to LangSmith."""
    
    # `evaluate` calls the target with just the example's inputs, so bind the settings here.
    def target(inputs: dict) -> dict:
        return run_agent_on_stream(context_settings, inputs)

    results = evaluate(target, data=DATASET_NAME, evaluators=EVALUATORS, experiment_prefix=prefix, max_concurrency=2)
    
    return average_scores(results)
    
print("baseline:", run_experiment(prefix="baseline", context_settings=MyAgentContext()))

## Step 5: Improve the alignment (3 min)

Getting an agent to score like your analysts takes a few rounds. Change one thing, run the evaluation again,
and compare the two experiments side by side in LangSmith.

Here we raise the confidence threshold in `tools/decisions.py`, so the agent asks a human more often. Watch what
happens to `review_routing`. Then try your own change, for example to the scoring guide in `instructions.md`.
`build_agent()` reads that file every time, so you don't need to restart anything.

In [ ]:
print("threshold-0.8:", run_experiment(prefix="threshold-0.8", context_settings=MyAgentContext(confidence_threshold=0.8)))

## Done!

You now have a repeatable way to check whether a change makes the agent score more like your analysts.

But the reference labels only cover the transactions analysts have already labelled. In **Module 3**, humans
review new transactions in an annotation queue, and their corrections feed back into this dataset.